# Collaborative Filtering Movie Recommender System


<div style="text-align: center;">
  <a href="https://colab.research.google.com/github/MinooSdpr/Machine-Learning-101/blob/main/Session%2017/17_3%20-%20Keras%20Project%20Exercise.ipynb">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab" />
  </a>
  &nbsp;
  <a href="https://github.com/MinooSdpr/Machine-Learning-101/blob/main/Session%2017/17_3%20-%20Keras%20Project%20Exercise.ipynb">
    <img src="https://img.shields.io/badge/Open%20in-GitHub-24292e?logo=github&logoColor=white" alt="Open In GitHub" />
  </a>
</div>

In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
import warnings


ratings_url = 'https://files.grouplens.org/datasets/movielens/ml-100k/u.data'


ratings = pd.read_csv(ratings_url, sep='\t', names=['user_id', 'movie_id', 'rating', 'timestamp'])

ratings.head()

,user_id,movie_id,rating,timestamp
0,196,242,3,881250949
1,186,302,3,891717742
2,22,377,1,878887116
3,244,51,2,880606923
4,166,346,1,886397596


In [2]:
movies_url = 'https://files.grouplens.org/datasets/movielens/ml-100k/u.item'


movies = pd.read_csv(movies_url, sep='|', encoding='latin-1', usecols=[0, 1], names=['movie_id', 'title'], header=None)

ratings = pd.merge(ratings, movies, on='movie_id')
ratings.head()

,user_id,movie_id,rating,timestamp,title
0,196,242,3,881250949,Kolya (1996)
1,186,302,3,891717742,L.A. Confidential (1997)
2,22,377,1,878887116,Heavyweights (1994)
3,244,51,2,880606923,Legends of the Fall (1994)
4,166,346,1,886397596,Jackie Brown (1997)


## Create User-Item Matrix


In [3]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(ratings, test_size=0.2, random_state=42)

train_matrix = train_df.pivot_table(index='user_id', columns='title', values='rating')
train_matrix_filled = train_matrix.fillna(0)

train_matrix_filled.head()

title,'Til There Was You (1997),1-900 (1994),101 Dalmatians (1996),12 Angry Men (1957),187 (1997),2 Days in the Valley (1996),"20,000 Leagues Under the Sea (1954)",2001: A Space Odyssey (1968),3 Ninjas: High Noon At Mega Mountain (1998),"39 Steps, The (1935)",...,Yankee Zulu (1994),Year of the Horse (1997),You So Crazy (1994),Young Frankenstein (1974),Young Guns (1988),Young Guns II (1990),"Young Poisoner's Handbook, The (1995)",Zeus and Roxanne (1997),unknown,Á köldum klaka (Cold Fever) (1994)
user_id,,,,,,,,,,,,,,,,,,,,,
1,0.0,0.0,2.0,5.0,0.0,0.0,0.0,4.0,0.0,0.0,...,0.0,0.0,0.0,5.0,3.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,2.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,0.0,0.0,2.0,0.0,0.0,0.0,0.0,4.0,0.0,0.0,...,0.0,0.0,0.0,4.0,0.0,0.0,0.0,0.0,4.0,0.0


## User-Based Collaborative Filtering

In this approach, we compute the similarity between users based on their movie ratings. Once we find users who are similar to a target user, we recommend movies that those similar users liked but the target user hasn't seen yet.

We'll use **cosine similarity** to measure user similarity.

### Understanding Cosine Similarity

**Cosine similarity** is a metric used to measure how similar two vectors are, regardless of their magnitude. It is commonly used in recommender systems to determine how similar users or items are based on their rating patterns.

In the context of user-based collaborative filtering:

- Each user is represented as a **vector of movie ratings**.
- Cosine similarity measures the **cosine of the angle** between two users' rating vectors.
- A smaller angle (closer to 1 cosine similarity) means the users have rated movies in a more similar way.

### Cosine Similarity Formula:

$$
\text{cosine\_similarity}(A, B) = \frac{A \cdot B}{\|A\| \|B\|}
$$

Where:
- $ A \cdot B$ is the dot product of the two vectors.
- $\|A\|$ and $\|B\|$ are the magnitudes (Euclidean norms) of the vectors.

### Interpretation of Cosine Similarity:
- **1** → Users are perfectly similar in their rating behavior.
- **0** → No similarity (orthogonal vectors).
- **-1** → Perfectly opposite rating behavior (rare in practice with non-negative data).

Cosine similarity is useful because it focuses on the **direction** of the ratings, not their **absolute values**, which helps when users have different rating scales (e.g., one user tends to rate everything highly, another rates more strictly).



In [4]:
user_similarity = cosine_similarity(train_matrix_filled)
user_similarity_df = pd.DataFrame(user_similarity, index=train_matrix.index, columns=train_matrix.index)
user_similarity_df.round(2).head()

user_id,1,2,3,4,5,6,7,8,9,10,...,934,935,936,937,938,939,940,941,942,943
user_id,,,,,,,,,,,,,,,,,,,,,
1,1.00,0.14,0.03,0.03,0.29,0.33,0.32,0.28,0.08,0.28,...,0.28,0.09,0.20,0.14,0.13,0.09,0.22,0.08,0.11,0.33
2,0.14,1.00,0.12,0.17,0.09,0.16,0.10,0.09,0.15,0.13,...,0.14,0.27,0.33,0.33,0.24,0.15,0.23,0.12,0.17,0.10
3,0.03,0.12,1.00,0.35,0.00,0.09,0.03,0.05,0.06,0.05,...,0.02,0.02,0.16,0.08,0.11,0.02,0.10,0.02,0.13,0.01
4,0.03,0.17,0.35,1.00,0.01,0.05,0.08,0.14,0.06,0.04,...,0.04,0.04,0.09,0.17,0.10,0.00,0.15,0.11,0.11,0.03
5,0.29,0.09,0.00,0.01,1.00,0.17,0.30,0.19,0.04,0.17,...,0.28,0.10,0.09,0.07,0.10,0.05,0.20,0.15,0.10,0.25


## Recommend Movies for a Target User

Now, let's pick a target user and recommend movies based on ratings from similar users.

Steps:
1. Choose a target user.
2. Find the most similar users to the target user.
3. Aggregate movie ratings from those similar users.
4. Recommend movies the target user hasn't rated yet.


In [5]:
def predict_user_based(user_id, movie_title, k=5):
    if movie_title not in train_matrix.columns or user_id not in user_similarity_df.index:
        return np.nan

    similar_users = user_similarity_df[user_id].drop(user_id, errors='ignore')
    top_k_users = similar_users.sort_values(ascending=False).head(k)

    movie_ratings = train_matrix.loc[top_k_users.index, movie_title]
    valid = movie_ratings[movie_ratings.notna()]
    valid_similarities = top_k_users.loc[valid.index]

    if valid.empty:
        return np.nan

    weighted_sum = np.dot(valid, valid_similarities)
    similarity_sum = valid_similarities.sum()

    return weighted_sum / similarity_sum if similarity_sum != 0 else np.nan


## Recommending formula for colaborative filtering


$$
\text{weighted\_rating}_m = \sum_{i=1}^{N} \left( \text{similarity}(u, i) \times \text{rating}_{i, m} \right)
$$
   where:

   * $u$ is the target user
   * $i$ is one of the top-N similar users
   * $m$ is a movie

### Final Recommendation Score:

To get the **final predicted rating** for each movie:


$$
\text{predicted\_rating}_m = \frac{\sum_{i=1}^{N} \left( \text{similarity}(u, i) \times \text{rating}_{i, m} \right)}{\sum_{i=1}^{N} \text{similarity}(u, i)}
$$

This gives us a list of movies scored by how likely the target user would enjoy them, based on the opinions of similar users.


In [6]:
from sklearn.metrics import mean_squared_error
from math import sqrt

def evaluate_rmse(test_df, k=5):
    predictions = []
    actuals = []

    for _, row in test_df.iterrows():
        pred = predict_user_based(row['user_id'], row['title'], k)
        if not np.isnan(pred):
            predictions.append(pred)
            actuals.append(row['rating'])

    rmse = sqrt(mean_squared_error(actuals, predictions))
    print(f"RMSE (k={k}): {rmse:.4f}")
    return rmse

evaluate_rmse(test_df, k=5)

RMSE (k=5): 1.1608


1.1608460574931732

In [7]:
from sklearn.metrics import pairwise_distances

cosine_distances = pairwise_distances(train_matrix_filled, metric='cosine')
user_similarity = 1 - cosine_distances

In [8]:
user_similarity[:5]

array([[1.        , 0.13828711, 0.03114461, ..., 0.08451645, 0.10611022,
        0.33060239],
       [0.13828711, 1.        , 0.1182149 , ..., 0.11881378, 0.1702726 ,
        0.09781364],
       [0.03114461, 0.1182149 , 1.        , ..., 0.02239009, 0.1310233 ,
        0.01407586],
       [0.02630753, 0.17012318, 0.35369641, ..., 0.11032395, 0.11351184,
        0.0323665 ],
       [0.28574869, 0.09418181, 0.        , ..., 0.14802815, 0.10101898,
        0.24752721]])

<div style="float:right;">
  <a href="https://github.com/MinooSdpr/Machine-Learning-101/blob/main/Session%2018/01%20-%20CNN%20.pptx"
     style="
       display:inline-block;
       padding:8px 20px;
       background-color:#414f6f;
       color:white;
       border-radius:12px;
       text-decoration:none;
       font-family:sans-serif;
       transition:background-color 0.3s ease;
     ">
    ▶️ Next
  </a>
</div>